# 👾 PixelCNN using Tensorflow distributions

In this notebook, we'll walk through the steps required to train your own PixelCNN on the fashion MNIST dataset using Tensorflow distributions

In [ ]:
%load_ext autoreload
%autoreload 2
# TensorBoard imports TensorFlow whenever it is installed, and TensorFlow and PyTorch
# in one process can crash. This makes TensorBoard use its own TF-free version instead
import sys, types
sys.modules["tensorboard.compat.notf"] = types.ModuleType("tensorboard.compat.notf")

import numpy as np

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import os
os.environ["KERAS_BACKEND"] = "torch"  # Keras only loads the dataset; this keeps TensorFlow out of the PyTorch process
from keras import datasets

# PyTorch has no PixelCNN distribution like TensorFlow Probability, so we use
# a compact PixelCNN++ implementation that lives next to this notebook
from pixelcnn_pp_pytorch import PixelCNN

import sys
from pathlib import Path

# Make the repo root importable so `notebooks.utils_pytorch` can be found
REPO_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(REPO_ROOT))

from notebooks.utils_pytorch import display

# PyTorch needs to be told which device to use
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():  # Apple Silicon
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

## 0. Parameters <a name="parameters"></a>

In [ ]:
IMAGE_SIZE = 32
N_COMPONENTS = 5
EPOCHS = 10
BATCH_SIZE = 128

## 1. Prepare the data <a name="prepare"></a>

In [ ]:
# Load the data - Keras downloads it once (cached in ~/.keras/datasets)
(x_train, _), (_, _) = datasets.fashion_mnist.load_data()  # uint8 array of shape (60000, 28, 28)

In [ ]:
# Preprocess the data


def preprocess(imgs):
    # (N, 28, 28) -> (N, 1, 28, 28): PyTorch images are channels-first (NCHW)
    imgs = torch.tensor(imgs, dtype=torch.float32).unsqueeze(1)
    imgs = F.interpolate(
        imgs, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False
    )
    # Keep pixel values in [0, 255], rounded to whole numbers because the
    # PixelCNN models discrete pixel values
    return imgs.round()


input_data = preprocess(x_train)

In [ ]:
# Show some items of clothing from the training set
display(input_data)

## 2. Build the PixelCNN <a name="build"></a>

In [ ]:
# Define a Pixel CNN network (see pixelcnn_pp_pytorch.py). Like the TFP version,
# it is a distribution over images with log_prob() and sample() methods
dist = PixelCNN(
    image_shape=(1, IMAGE_SIZE, IMAGE_SIZE),  # channels first
    num_resnet=1,
    num_hierarchies=2,
    num_filters=32,
    num_logistic_mix=N_COMPONENTS,
    dropout_p=0.3,
).to(device)

print(dist)
print(f"Total params: {sum(p.numel() for p in dist.parameters()):,}")

# The loss is the negative log-likelihood of the images (computed in the training loop):
# loss = -dist.log_prob(images).mean()

## 3. Train the PixelCNN <a name="train"></a>

In [ ]:
# The model's weights are the PixelCNN's parameters
optimizer = torch.optim.Adam(dist.parameters(), lr=0.001)

In [ ]:
writer = SummaryWriter(log_dir="./logs")


# Samples images (called at the end of each epoch in the training loop)
class ImageGenerator:
    def __init__(self, num_img):
        self.num_img = num_img

    def generate(self):
        return dist.sample(self.num_img).cpu().numpy()


img_generator_callback = ImageGenerator(num_img=2)

In [ ]:
train_loader = DataLoader(input_data, batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(EPOCHS):
    dist.train()  # enables dropout
    total_loss = 0.0
    for images in train_loader:
        images = images.to(device)

        loss = -dist.log_prob(images).mean()  # negative log-likelihood per image

        optimizer.zero_grad()  # reset gradients from the previous step
        loss.backward()  # backpropagation
        optimizer.step()  # update the weights

        total_loss += loss.item()

    epoch_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {epoch_loss:.4f}")
    writer.add_scalar("loss", epoch_loss, epoch)

    # What the Keras ImageGenerator callback did at the end of each epoch
    generated_images = img_generator_callback.generate()
    display(
        generated_images,
        n=img_generator_callback.num_img,
        save_to="./output/generated_img_%03d.png" % (epoch),
    )

writer.close()

## 4. Generate images <a name="generate"></a>

In [ ]:
generated_images = img_generator_callback.generate()

In [ ]:
display(generated_images, n=img_generator_callback.num_img)